In [ ]:
#@title Install dependencies (Colab)
!pip install -q rdkit networkx

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, QED
from rdkit.Chem import rdmolops
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


In [ ]:
from rdkit import Chem

def canonicalize_smiles(smiles):
    if pd.isna(smiles):
        return None

    smiles = str(smiles).strip()

    if not smiles:
        return None

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    return Chem.MolToSmiles(mol, canonical=True)

In [ ]:
#@title Auto-fetch input data from GitHub (no manual upload needed)
import os, urllib.request
RAW_CSV_URL = "https://raw.githubusercontent.com/anasuyapanyala-ux/BACE1-QSAR-Model/main/BACE1_raw_chembl.csv"
RAW_CSV_LOCAL = "BACE1_raw_chembl.csv"
if not os.path.exists(RAW_CSV_LOCAL):
    urllib.request.urlretrieve(RAW_CSV_URL, RAW_CSV_LOCAL)
    print(f"Downloaded {RAW_CSV_LOCAL} from GitHub")
else:
    print(f"{RAW_CSV_LOCAL} already present")

In [ ]:
TARGET_NAME = "BACE1"

# Load the manually downloaded ChEMBL bioactivity CSV
raw_df = pd.read_csv(
    "BACE1_raw_chembl.csv",
    sep=",",
    quotechar='"',
    low_memory=False
)

print(f"Loaded {len(raw_df)} raw records for {TARGET_NAME}")

raw_df.columns = raw_df.columns.str.strip()
print(raw_df.columns.tolist())

In [ ]:
def clean_bioactivity_df(df):
    df = df.copy()
    df = df[df['Standard Type'] == 'IC50']
    df = df[df['Standard Relation'] == "'='"]  # ChEMBL export often quotes this
    df = df[df['Standard Units'] == 'nM']
    df = df[['Smiles', 'Standard Value']].dropna()
    df.columns = ['smiles_raw', 'standard_value']
    df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
    df = df.dropna(subset=['standard_value'])
    df = df[df['standard_value'] > 0]

    df['SMILES'] = df['smiles_raw'].apply(canonicalize_smiles)
    df = df.dropna(subset=['SMILES'])

    df['Activity'] = -np.log10(df['standard_value'] * 1e-9)
    df = df.groupby('SMILES', as_index=False)['Activity'].median()
    return df

data = clean_bioactivity_df(raw_df)
print(f"{TARGET_NAME}: {len(raw_df)} raw -> {len(data)} cleaned unique compounds")
data.to_csv(f'BACE1_QSAR.csv', index=False)
data.head()

---
## Section 1: Descriptor Generation



### 1.1 Topological Descriptors
Wiener Index, Zagreb Index, TPSA, Balaban Index, Kappa1-3 (Kier-Hall chi indices).


In [ ]:
#@title Calculate Specific Topological Descriptors

def wiener_index(mol):
    g = nx.Graph(rdmolops.GetAdjacencyMatrix(mol))
    return nx.wiener_index(g)

def zagreb_index(mol):
    adj_matrix = rdmolops.GetAdjacencyMatrix(mol)
    degrees = adj_matrix.sum(axis=0)
    return sum(degrees**2)

def tpsa(mol):
    return Descriptors.TPSA(mol)

def kier_hall_chi_indices(mol):
    return {
        'Kappa1': Descriptors.Kappa1(mol),
        'Kappa2': Descriptors.Kappa2(mol),
        'Kappa3': Descriptors.Kappa3(mol),
    }

def balaban_index(mol):
    return Descriptors.BalabanJ(mol)

def calculate_topological_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    descriptors = {
        'WienerIndex': wiener_index(mol),
        'ZagrebIndex': zagreb_index(mol),
        'TPSA': tpsa(mol),
        'BalabanIndex': balaban_index(mol),
    }
    descriptors.update(kier_hall_chi_indices(mol))
    return descriptors

descriptor_data = []
for index, row in data.iterrows():
    smiles = row['SMILES']
    activity = row['Activity']
    descriptors = calculate_topological_descriptors(smiles)
    if descriptors is not None:
        descriptors['SMILES'] = smiles
        descriptors['Activity'] = activity
        descriptor_data.append(descriptors)

descriptor_df = pd.DataFrame(descriptor_data)
descriptor_df.to_csv(f'BACE1_Topological_descriptors.csv', index=False)
print(f"Topological descriptors saved: {len(descriptor_df)} compounds")


### 1.2 Lipinski Descriptors
MW, LogP, NumHDonors, NumHAcceptors, NumRotatableBonds, NumRings.


In [ ]:
#@title Calculate Lipinski's Descriptors

def lipinski_descriptors(mol):
    if mol is None:
        return None
    return {
        'MolecularWeight': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'NumRings': Descriptors.RingCount(mol),
    }

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return lipinski_descriptors(mol)

descriptor_data = []
for index, row in data.iterrows():
    smiles = row['SMILES']
    activity = row['Activity']
    descriptors = calculate_descriptors(smiles)
    if descriptors is not None:
        descriptors['SMILES'] = smiles
        descriptors['Activity'] = activity
        descriptor_data.append(descriptors)

descriptor_df = pd.DataFrame(descriptor_data)
descriptor_df.to_csv(f'BACE1_lipinski_descriptors.csv', index=False)
print(f"Lipinski descriptors saved: {len(descriptor_df)} compounds")


### 1.4 QED (Quantitative Estimation of Drug-likeness) Score


In [ ]:
#@title Calculate QED Score

def qed_score(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return QED.qed(mol)

qed_data = []
for index, row in data.iterrows():
    smiles = row['SMILES']
    activity = row['Activity']
    qed_value = qed_score(smiles)
    if qed_value is not None:
        qed_data.append({'SMILES': smiles, 'Activity': activity, 'QED': qed_value})

qed_df = pd.DataFrame(qed_data)
qed_df.to_csv(f'BACE1_qed_scores.csv', index=False)
print(f"QED scores saved: {len(qed_df)} compounds")


## Section 2: Merge All Descriptor Sets



In [ ]:
df_qed = pd.read_csv(f'{TARGET_NAME}_qed_scores.csv')
df_lipinski = pd.read_csv(f'{TARGET_NAME}_lipinski_descriptors.csv').drop(columns=['Activity'])
df_topological = pd.read_csv(f'{TARGET_NAME}_Topological_descriptors.csv').drop(columns=['Activity'])

merged_df = pd.merge(df_qed, df_lipinski, on='SMILES', how='outer')
merged_df = pd.merge(merged_df, df_topological, on='SMILES', how='outer')

merged_df.to_csv(f'{TARGET_NAME}_merged_descriptors.csv', index=False)
print(f"Merged (2D-only, no 3D): {len(merged_df)} compounds, {merged_df.shape[1]} columns")
merged_df.head()